# Fuzzy Membership Finetuning

Fine-tunes a small BERT model on soft (fuzzy) cluster membership vectors using
the `fuzzyfinetuning_crossentropy.py` script from the RecsysUpgrade repo.

Reads the membership matrix saved by the clustering notebook (`fkmeans_memberships.csv`
and `fkmeans_clusters.csv`) from the same `CLUSTERS_OUTPUT_DIR`.

### Step 1: Setup (drive mounting, paths)

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ── Match these paths to clustering_comparison.ipynb ──────────────────────────
DRIVE_BASE          = "/content/drive/MyDrive/S2026/Spotify Playlist Data/Processed Data/calced_embeddings"
PROJECT_BASE        = "/content/drive/MyDrive/S2026/CS 274/Playlist-Recommender"
EMBEDDINGS_PKL      = f"{DRIVE_BASE}/mean_pooled_embeddings.pkl"
CLUSTERS_OUTPUT_DIR = f"{PROJECT_BASE}/new clusters"
# ──────────────────────────────────────────────────────────────────────────────

FINETUNED_MODEL_DIR = f"{PROJECT_BASE}/fuzzy_finetuned_model"
REPO_DIR            = "/content/RecsysUpgrade"

os.makedirs(FINETUNED_MODEL_DIR, exist_ok=True)
print("Paths configured.")

Mounted at /content/drive
Paths configured.


### Step 2: Clone repo and install requirements

In [2]:
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/siddmohanty111/RecsysUpgrade.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print("Repo ready:", REPO_DIR)

Cloning into '/content/RecsysUpgrade'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 85 (delta 40), reused 56 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 1.44 MiB | 7.74 MiB/s, done.
Resolving deltas: 100% (40/40), done.
Repo ready: /content/RecsysUpgrade


In [3]:
%pip install -r {REPO_DIR}/requirements.txt -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


### Step 3: Load membership matrix and playlist titles

In [4]:
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# fkmeans_clusters.csv was saved alongside fkmeans_memberships.csv from the same `pids` list,
# so row i in memberships corresponds to row i (pid) in clusters.
clusters_df = pd.read_csv(
    os.path.join(CLUSTERS_OUTPUT_DIR, "fkmeans_clusters.csv"),
    dtype={"pid": str},
)
pids_ordered = clusters_df["pid"].tolist()

memberships_df = pd.read_csv(os.path.join(CLUSTERS_OUTPUT_DIR, "fkmeans_memberships.csv"))
membership_cols = [c for c in memberships_df.columns if c.startswith("cluster_")]
num_clusters = len(membership_cols)
print(f"Membership matrix: {len(pids_ordered)} playlists × {num_clusters} clusters")

# Load playlist titles from the embeddings pkl (same source as the clustering notebook)
with open(EMBEDDINGS_PKL, "rb") as f:
    raw_data = pickle.load(f)

playlist_titles = (
    raw_data.get("playlist_titles", {})
    if isinstance(raw_data, dict)
    else {}
)
print(f"Playlist titles available: {len(playlist_titles)}")

Membership matrix: 1000001 playlists × 50 clusters
Playlist titles available: 1000001


### Step 4: Build dataset and train/val split

In [5]:
# Build a DataFrame with the two columns expected by fuzzyfinetuning_crossentropy.run():
#   "Playlist Title"  — string title
#   "Cluster Labels"  — string repr of a Python list, e.g. "[0.12, 0.03, ...]"
#                       (parsed by ast.literal_eval inside the finetuning script)

data_df = pd.DataFrame({
    "Playlist Title": [playlist_titles.get(pid, "") for pid in pids_ordered],
    "Cluster Labels": memberships_df[membership_cols].values.tolist(),
})

# Convert each membership list to its string representation
data_df["Cluster Labels"] = data_df["Cluster Labels"].apply(str)

# Drop playlists with no title — they carry no signal for the model
data_df = data_df[data_df["Playlist Title"].str.strip() != ""].reset_index(drop=True)
print(f"Dataset after dropping untitled playlists: {len(data_df)}")

# 90 / 10 train–val split
train_df, val_df = train_test_split(data_df, test_size=0.1, random_state=42)

TRAIN_CSV = os.path.join(CLUSTERS_OUTPUT_DIR, "fuzzy_train.csv")
VAL_CSV   = os.path.join(CLUSTERS_OUTPUT_DIR, "fuzzy_val.csv")

train_df.to_csv(TRAIN_CSV, index=False)
val_df.to_csv(VAL_CSV,   index=False)
print(f"Train: {len(train_df)}  Val: {len(val_df)}")
print(f"Saved to {CLUSTERS_OUTPUT_DIR}")

Dataset after dropping untitled playlists: 1000001
Train: 900000  Val: 100001
Saved to /content/drive/MyDrive/S2026/CS 274/Playlist-Recommender/new clusters


### Step 5: Load finetuning module and run

In [6]:
import importlib.util

ft_path = os.path.join(
    REPO_DIR, "PlaylistRecsysUpgrade", "finetuning", "fuzzyfinetuning_crossentropy.py"
)
spec   = importlib.util.spec_from_file_location("fuzzyfinetuning_crossentropy", ft_path)
fuzzy_ft = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fuzzy_ft)
print("fuzzyfinetuning_crossentropy loaded.")

fuzzyfinetuning_crossentropy loaded.


In [7]:
# prajjwal1/bert-tiny  — 4.4 M params, trains ~10× faster than MiniLM on Colab.
# Swap to "prajjwal1/bert-mini" or "distilbert-base-uncased" for better quality.
fuzzy_ft.run(
    train_csv=TRAIN_CSV,
    val_csv=VAL_CSV,
    output_dir=FINETUNED_MODEL_DIR,
    model_name="prajjwal1/bert-tiny",
    batch_size=128,
    epochs=20,
    learning_rate=3e-4,
    warmup_steps=200,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.encoder.layer.0.output.LayerNorm.beta', 'bert.encoder.layer.0.output.LayerNorm.gamma', 'bert.encoder.layer.1.attention.output.LayerNorm.beta', 'bert.encoder.layer.1.attention.output.LayerNorm.gamma', 'bert.en

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved to /content/drive/MyDrive/S2026/CS 274/Playlist-Recommender/fuzzy_finetuned_model
